# CSE 151B Competition — ARC Challenge (Ollama Version)

Welcome to the **CSE 151B Spring 2026 ARC Challenge Notebook**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `pip`
2. Loading the ARC Challenge dataset from HuggingFace
3. Running inference with a local model via **Ollama**
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The ARC Challenge dataset contains science questions **with** answers so you can measure accuracy locally.  
For the private test set used for the leaderboard, skip evaluation and submit the raw responses.


## 1. Environment Setup

We use `pip` to install all required packages.

Requirements:
- `ollama` — Python client for the Ollama inference server
- `datasets` — HuggingFace datasets library to download ARC
- `tqdm` — progress bars

**Before running inference, make sure Ollama is running:**
```
ollama serve
```

**Pull the model once (one-time setup):**
```
ollama pull llama3.2:1b
```

> **Note:** Change `MODEL_ID` in Section 2 to any model you have installed locally.

### Run the cell below to install dependencies.

In [ ]:
!pip install ollama tqdm datasets

## 2. Imports & Configuration

All key settings are collected in one place.
- `MODEL_ID` — the Ollama model to use (must be pulled first)
- `DATA_PATH` — where the ARC Challenge JSONL will be saved locally
- `OUTPUT_PATH` — where per-question results will be written
- `MAX_TOKENS` — maximum tokens the model may generate per response
- `SAVE_EVAL` — set to `False` when running on the private test set (no ground-truth)

In [ ]:
import json
import re
import sys
from pathlib import Path
from typing import Optional

import ollama
from tqdm import tqdm

# ── Configuration ──────────────────────────────────────────────────────────────
MODEL_ID    = "llama3.2:1b"               # Change to any installed Ollama model
DATA_PATH   = "data/arc_challenge_test.jsonl"
OUTPUT_PATH = "results/arc_results.jsonl"
MAX_TOKENS  = 512
SAVE_EVAL   = True                          # Set False for private test set

## 3. Load the Dataset

The ARC Challenge dataset is downloaded from HuggingFace and saved as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | Flat list of answer choice strings |
| `answer` | Ground-truth answer letter (`A`, `B`, `C`, or `D`) |

ARC sometimes uses numeric labels (`"1"`, `"2"`, `"3"`, `"4"`) instead of letter labels — the loader remaps these automatically to `A`/`B`/`C`/`D`.

In [ ]:
def download_arc_dataset(save_path: str):
    """Download ARC Challenge test set from HuggingFace and save as JSONL."""
    try:
        from datasets import load_dataset
    except ImportError:
        print("ERROR: 'datasets' library not found. Run: pip install datasets")
        sys.exit(1)

    print("Downloading ARC Challenge dataset from HuggingFace...")
    ds = load_dataset("allenai/ai2_arc", "ARC-Challenge", split="test")

    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    ds.to_json(save_path)
    print(f"Saved to {save_path}\n")


def load_arc_dataset(path: str) -> list:
    """
    Load ARC Challenge JSONL and convert to the format this script expects.

    ARC format:
      {
        "id": "Mercury_7175875",
        "question": "Which factor ...",
        "choices": { "text": ["A", "B", "C", "D"], "label": ["A","B","C","D"] },
        "answerKey": "B"
      }

    Converted to:
      {
        "id": "Mercury_7175875",
        "question": "Which factor ...",
        "options": ["choice A text", "choice B text", ...],
        "answer": "B"
      }
    """
    data = []
    for line in open(path):
        item = json.loads(line)
        choices = item["choices"]

        # Re-map answer key: ARC sometimes uses "1","2","3","4" instead of "A","B","C","D"
        label_to_letter = {
            label: chr(65 + i)
            for i, label in enumerate(choices["label"])
        }
        answer_letter = label_to_letter.get(item["answerKey"], item["answerKey"])

        data.append({
            "id":       item["id"],
            "question": item["question"],
            "options":  choices["text"],     # flat list of answer strings
            "answer":   answer_letter,       # single letter: A, B, C, or D
        })
    return data


# Download if not already present, then load
if not Path(DATA_PATH).exists():
    download_arc_dataset(DATA_PATH)

print(f"Loading dataset from {DATA_PATH}...")
data = load_arc_dataset(DATA_PATH)
print(f"Loaded {len(data)} questions\n")

# Preview a sample question
sample = data[0]
print("── Sample question ──")
print(f"  Q : {sample['question']}")
for i, opt in enumerate(sample["options"]):
    print(f"  {chr(65+i)} : {opt}")
print(f"  Answer: {sample['answer']}")

## 4. Prompt Construction

Since all ARC Challenge questions are multiple-choice, we use a single MCQ system prompt:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [ ]:
# ── System Prompts ─────────────────────────────────────────────────────────────
SYSTEM_PROMPT_MCQ = (
    "You are an expert at answering science and reasoning questions. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)


def build_prompt(question: str, options: list) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a MCQ question."""
    labels    = [chr(65 + i) for i in range(len(options))]
    opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
    return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"


# Verify with sample
sys_p, usr_p = build_prompt(sample["question"], sample["options"])
print("── Sample user prompt ──")
print(usr_p[:300], "...\n")

## 5. Run Inference with Ollama

We call Ollama's `chat` API for each question sequentially.  
Ollama runs locally and processes one question at a time.

Key sampling parameters:
- `temperature` — controls randomness (0.6 is a good default for reasoning tasks)
- `top_p` / `top_k` — nucleus and top-k sampling
- `num_predict` — equivalent to `max_tokens`

> **Tip:** To test on a small subset first, uncomment `data = data[:20]` below.

In [ ]:
def run_inference(data: list) -> list:
    """Run Ollama inference on all questions."""
    responses = []

    print(f"Running inference on {len(data)} questions with model '{MODEL_ID}'...")
    print("(Ollama runs one question at a time)\n")

    for item in tqdm(data, desc="Generating"):
        system, user = build_prompt(item["question"], item["options"])

        try:
            response = ollama.chat(
                model=MODEL_ID,
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user",   "content": user},
                ],
                options={
                    "num_predict":    MAX_TOKENS,
                    "temperature":    0.6,
                    "top_p":          0.95,
                    "top_k":          20,
                    "repeat_penalty": 1.0,
                },
            )
            text = response["message"]["content"].strip()
        except Exception as e:
            print(f"\nError on question id={item.get('id')}: {e}")
            text = ""

        responses.append(text)

    return responses


# To test on a small subset first, uncomment the line below:
# data = data[:20]
responses = run_inference(data)

# Preview first 3 responses
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

## 6. Score Responses

Since all ARC questions are MCQ, scoring is exact-match on the predicted letter:

1. Extract the predicted letter from `\boxed{X}`
2. Fall back to the last standalone uppercase letter in the response if no `\boxed{}` is found
3. Compare to the gold answer letter

Each result record contains `{id, gold, predicted, response, correct}`.

In [ ]:
def extract_letter(text: str) -> str:
    """Extract answer letter from \\boxed{X}, or fall back to last uppercase letter."""
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_results(data: list, responses: list) -> list:
    results = []
    for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
        predicted = extract_letter(response)
        correct   = predicted == item["answer"].strip().upper()

        results.append({
            "id":        item["id"],
            "gold":      item["answer"],
            "predicted": predicted,
            "response":  response,
            "correct":   correct,
        })
    return results


results = score_results(data, responses)
print(f"Scoring complete. {len(results)} results.")

## 7. Summary

Print overall accuracy on the ARC Challenge test set.

In [ ]:
total   = len(results)
correct = sum(r["correct"] for r in results)
acc     = correct / total * 100 if total else 0.0

print("=" * 50)
print("EVALUATION RESULTS — ARC Challenge")
print("=" * 50)
print(f"  Correct : {correct:4d} / {total:4d}  ({acc:.2f}%)")
print("=" * 50)

## 8. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (`SAVE_EVAL = True`, public set — you have ground-truth):  
Each line: `{id, gold, predicted, response, correct}`

**Without evaluation** (`SAVE_EVAL = False`, private test set — no ground-truth available):  
Each line: `{id, predicted, response}` — omits `gold` and `correct`.

Toggle `SAVE_EVAL` at the top of the notebook accordingly.

In [ ]:
out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {
                "id":        r["id"],
                "gold":      r["gold"],
                "predicted": r["predicted"],
                "response":  r["response"],
                "correct":   r["correct"],
            }
        else:
            record = {
                "id":        r["id"],
                "predicted": r["predicted"],
                "response":  r["response"],
            }
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Model selection** — swap `MODEL_ID` to a larger or more capable Ollama model (e.g. `llama3.1:8b`, `qwen2.5:7b`)
- **Chain-of-thought** — modify the system prompt to encourage step-by-step reasoning before selecting the answer letter

Good luck!